In [1]:

import numpy as np
from sklearn.datasets import load_iris
import pandas as pd
iris = load_iris()
from sklearn.svm import SVC

In [ ]:
df = pd.DataFrame(iris.data,columns=iris.feature_names)

In [ ]:
df["name_c"] = iris.target
df["name"] = df.name_c.apply(lambda x : iris.target_names[x])

In [ ]:
df.drop("name_c",axis=1,inplace=True)

In [ ]:
df.head(2)

In [ ]:
from sklearn.model_selection import train_test_split,GridSearchCV,RandomizedSearchCV
X_train, X_test, y_train, y_test = train_test_split(df.drop("name",axis=1),df.name, test_size = 0.2)


In [ ]:
model = SVC(kernel="rbf",C=10,gamma="auto")
model.fit(X_train,y_train)
model.score(X_test,y_test)

In [ ]:
from sklearn.model_selection import cross_val_score
cross_val_score(SVC(kernel="linear",C=10,gamma="auto"),df.drop("name",axis=1),df.name,cv=5)

In [ ]:
cross_val_score(SVC(kernel="rbf",C=10,gamma="auto"),df.drop("name",axis=1),df.name,cv=5)

In [ ]:
cross_val_score(SVC(kernel="rbf",C=20,gamma="auto"),df.drop("name",axis=1),df.name,cv=5)

In [ ]:
kernels = ["linear","poly","rbf","sigmoid"]
C = [10,20,30,40,50]
gamma = [0.1,0.2,0.3,0.4,0.5,"auto"]
avg_score = {}
for k in kernels:
    for c in C:
        for g in gamma:
            cv_score = cross_val_score(SVC(kernel=k,C=c,gamma=g),df.drop("name",axis=1),df.name,cv=5)
            avg_score[k +"_"+ str(c) +"gamma"+ str(g)] = np.average(cv_score)

avg_score

In [ ]:
df1 = pd.DataFrame(avg_score.items(),columns=["combinations","score"])
df1.score.unique()

In [ ]:
gcv = GridSearchCV(SVC(),{
    "kernel" : ["linear","poly","rbf","sigmoid"],
    "C": [10,20,30,40,50],
    "gamma": [0.1,0.2,0.3,0.4,0.5,"auto"]
},cv=5)
gcv.fit(df.drop("name",axis=1),df.name)
gcv.cv_results_

In [ ]:
df2 = pd.DataFrame(gcv.cv_results_)

In [ ]:
df2.head(5)

In [ ]:
gcv.get_params

In [ ]:
rcv = RandomizedSearchCV(SVC(),{
    "kernel" : ["linear","poly","rbf","sigmoid"],
    "C": [10,20,30,40,50],
    "gamma": [0.1,0.2,0.3,0.4,0.5,"auto"]
},cv=5,return_train_score=False,n_iter=5)
rcv.fit(df.drop("name",axis=1),df.name)
pd.DataFrame(rcv.cv_results_)["mean_test_score"]

In [ ]:
rcv.best_score_

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
model_param = {
    "svm" : {
        "model" : SVC(),
        "param" :{
            "kernel" : ["linear","poly","rbf","sigmoid"],
            "C": [10,20,30,40,50],
            "gamma": [0.1,0.2,0.3,0.4,0.5,"auto"]
        }
    },
    "random_forest" : {
        "model" : RandomForestClassifier(),
        "param" :{
            "n_estimators" : [1,5,10]
        }
    },
    "logitic" : {
        "model" : LogisticRegression(),
        "param" :{
            "C": [10,20,30,40,50],
        }
    }
}

In [ ]:
model_param.items()

In [ ]:
score = []
for model_name,p in model_param.items() :
    clf = GridSearchCV(p["model"],p["param"],cv=3,return_train_score=False)
    clf.fit(df.drop("name",axis=1),df.name)
    score.append({
        "model" : model_name,
        "best_score" : clf.best_score_,
        "best_params" : clf.best_params_,
    })

In [ ]:
df = pd.DataFrame(score,columns=["model","best_score","best_params"])

In [ ]:
df